# A2a -- multi-gene granule reanalysis

Analysis **A2a** of `plans/Round2_response_analysis_plan.md`: the stratification by granule
complexity that Reviewer #2 says *"has not been met"*.

Subset the published pair-1 granules to those with **>= 3 unique genes**, then rerun the whole
published downstream chain on that subset -- subtyping, WT-vs-AD density, neuropil microdomains,
microdomain DE -- and show the structure persists at reduced n. A secondary section stratifies
**all** granules by read count, to answer the other half of the reviewer's sentence ("not a
low-count artifact").

### Which column is the complexity

**Not `granules.parquet["comp"]`.** `mcDETECT_package/mcDETECT/model.py:102` restricts the
transcript frame to `gnl_genes` before detection, so `model.py:264-266`
(`other_comp = len(other_trans[detect_col].unique())`, `total_comp = 1 + other_comp`) counts
**distinct granule markers**, capped at 20 -- empirical max 19, mean 2.53 (WT) / 2.44 (AD). It is
also never recomputed after `merge_sphere()`, since `_remove_overlaps` updates only
`sphere_x/y/z/r` or copies the other sphere's row wholesale.

The complexity used here comes from `profile()` (`model.py:432-473`), which counts **all** panel
genes inside the sphere: `n_genes = (layers["counts"] > 0).sum(axis=1)`. Section 1 exports the
`comp`-vs-`n_genes` cross-tab so the difference is on the record.

### No re-detection

`profile()` is independent per sphere, so subsetting rows of the published granule x gene matrix
is identical to re-profiling the retained spheres -- asserted in section 7. Nothing here reruns
detection or touches `transcripts.parquet`.

### Two manual pauses

This notebook is designed to be run **twice**. On the first pass leave `MANUAL_SUBTYPE_MAPPING`
empty and `SUBDOMAIN_PAIRS` at its placeholder: section 3 stops at the heatmap and section 5
still produces its subdomain map. Read the two figures, fill both blocks in, then rerun end to
end. Nothing in those sections is cached -- an output-exists check would skip exactly the second
pass.

**Run this notebook from `R2_revision/sparsity_structure/`.**

## 0. Setup

In [1]:
import json
import sys
from pathlib import Path

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture

from mcDETECT.downstream import spot_embedding

sys.path.insert(0, str(Path.cwd()))          # run this notebook from R2_revision/sparsity_structure/
import a2_config as C
import a2_common as A2

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# -------------------- runtime gates -------------------- #
MIN_UNIQUE_GENES = C.MIN_UNIQUE_GENES        # 3; drop to 2 only if section 1 says retention is poor
MAX_GRANULES = None                          # e.g. 200_000 for a fast dry run
VALIDATE = False                             # section 7 correctness gates
RUN_NEURON_REFERENCE = False                 # the slow neuron-granule count histogram (section 5)

C.ensure_dirs()
OUT = C.A2A_MULTIGENE_DIR
OUT_STRATA = C.A2A_READSTRATA_DIR
print("writing to", OUT)

/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-p

writing to /Users/chenyang/Desktop/mcDETECT/R2_revision/sparsity_structure/output/a2a/multigene


## 1. Granule complexity and the multi-gene subset

Reads the published combined granule profile (WT+AD, 1,080,146 x 290, built by
`code/4_post_detection.ipynb` cell 19) and derives per-granule read and unique-gene counts from
its raw `counts` layer.

Three outputs matter downstream:

* `complexity_summary.csv` / `complexity_histogram.parquet` -- the distributions (this supersedes
  Fig. R9, which the reviewer asked twice to see in the manuscript);
* `comp_vs_ngenes.parquet` -- the cross-tab showing what `comp` actually counted;
* the boolean `keep` mask that defines the subset for sections 2-5.

In [2]:
granule_adata = sc.read_h5ad(C.COMBINED_GRANULE_ADATA)
print(granule_adata)

genes_all = list(granule_adata.var_names)
nc_genes = list(pd.read_csv(C.nc_path("WT"))["Gene"])
nc_in_panel = [g for g in nc_genes if g in genes_all]
print(f"{len(genes_all)} panel genes, {len(nc_in_panel)} of {len(nc_genes)} negative controls "
      f"present in the panel")

# This panel has no blank probes -- the negative controls are real nuclear-enriched panel genes,
# so "unique genes" needs a stated denominator. Primary count excludes them; the all-panel count
# rides along as a sensitivity column.
counts = granule_adata.layers["counts"]
n_reads, n_genes_panel = A2.unique_gene_counts(counts, genes_all)
_, n_genes_nonNC = A2.unique_gene_counts(counts, genes_all, exclude_genes=nc_in_panel)

granule_adata.obs["n_reads"] = n_reads
granule_adata.obs["n_genes_panel"] = n_genes_panel
granule_adata.obs["n_genes_nonNC"] = n_genes_nonNC

complexity_col = "n_genes_nonNC" if C.EXCLUDE_NC_FROM_COMPLEXITY else "n_genes_panel"
print(f"complexity column = {complexity_col}")
print(granule_adata.obs[["n_reads", "n_genes_panel", "n_genes_nonNC", "comp"]].describe())

AnnData object with n_obs × n_vars = 1080146 × 290
    obs: 'global_x', 'global_y', 'global_z', 'layer_z', 'sphere_r', 'size', 'comp', 'in_soma_ratio', 'gene', 'nc_ratio', 'brain_area', 'global_y_new', 'global_x_new', 'granule_id', 'global_x_adjusted', 'global_y_adjusted', 'batch'
    var: 'genes'
    uns: 'log1p', 'pca', 'tsne'
    obsm: 'X_pca', 'X_tsne'
    varm: 'PCs'
    layers: 'counts'
290 panel genes, 19 of 19 negative controls present in the panel
complexity column = n_genes_nonNC
            n_reads  n_genes_panel  n_genes_nonNC          comp
count  1.080146e+06   1.080146e+06   1.080146e+06  1.080146e+06
mean   8.764174e+00   5.736557e+00   5.704109e+00  2.498914e+00
std    1.323629e+01   6.486286e+00   6.355053e+00  1.893946e+00
min    1.000000e+00   1.000000e+00   1.000000e+00  1.000000e+00
25%    3.000000e+00   2.000000e+00   2.000000e+00  1.000000e+00
50%    5.000000e+00   4.000000e+00   4.000000e+00  2.000000e+00
75%    9.000000e+00   7.000000e+00   7.000000e+00  3.0000

In [3]:
# Cross-check against the published Fig. R9 table. It was written with mc.profile(buffer=0.01)
# while granule_adata_tsne.h5ad was built with buffer=0.0, so a small disagreement is expected
# and is reported rather than papered over.
ref = []
for sample in C.SAMPLES:
    df = pd.read_parquet(C.mcdetect_reads_genes_path(sample))
    df["batch"] = C.dataset(sample)
    ref.append(df)
ref = pd.concat(ref, ignore_index=True)

chk = (granule_adata.obs[["batch", "granule_id", "n_reads", "n_genes_panel"]]
       .merge(ref, on=["batch", "granule_id"], how="left", validate="one_to_one"))
assert chk["reads_per_granule"].notna().all(), "granule_id join to the Fig. R9 table lost rows"

agree_reads = float((chk["n_reads"] == chk["reads_per_granule"]).mean())
agree_genes = float((chk["n_genes_panel"] == chk["unique_genes_per_granule"]).mean())
print(f"agreement with granule_reads_unique_genes_per_granule.parquet (buffer 0.00 vs 0.01): "
      f"reads {agree_reads:.4f}, unique genes {agree_genes:.4f}")
print("median reads/granule:", int(np.median(n_reads)),
      "| median unique genes/granule:", int(np.median(n_genes_panel)))
pd.DataFrame([{"agreement_reads": agree_reads, "agreement_unique_genes": agree_genes,
               "median_reads": float(np.median(n_reads)),
               "median_unique_genes_panel": float(np.median(n_genes_panel)),
               "median_unique_genes_nonNC": float(np.median(n_genes_nonNC)),
               "buffer_this_object": 0.0, "buffer_published_table": 0.01}]
             ).to_csv(OUT / "complexity_crosscheck.csv", index=False)

AssertionError: granule_id join to the Fig. R9 table lost rows

In [4]:
# What `comp` actually counted, versus the panel-wide unique-gene count.
ct = (pd.crosstab(granule_adata.obs["comp"].astype(int),
                  granule_adata.obs[complexity_col].astype(int))
        .stack().rename("count").reset_index())
ct.columns = ["comp", "n_genes", "count"]
ct.to_parquet(OUT / "comp_vs_ngenes.parquet", index=False)

disagree = float((granule_adata.obs["comp"].astype(int)
                  != granule_adata.obs[complexity_col].astype(int)).mean())
print(f"comp max = {int(granule_adata.obs['comp'].max())} (marker count, capped at "
      f"{len(C.SYN_GENES)})")
print(f"comp disagrees with {complexity_col} for {disagree:.1%} of granules")

comp max = 19 (marker count, capped at 20)
comp disagrees with n_genes_nonNC for 77.1% of granules


In [5]:
# Distributions, pre-binned for R.
summary_rows, hist_rows = [], []
for sample in C.SAMPLES:
    m = (granule_adata.obs["batch"] == C.dataset(sample)).to_numpy()
    for measure, col in [("n_reads", "n_reads"), ("n_genes", complexity_col),
                         ("sphere_r", "sphere_r"), ("size", "size")]:
        s, h = A2.record_distribution(granule_adata.obs.loc[m, col].to_numpy(), measure,
                                      C.HIST_BINS.get(measure, (0.0, 60.0, 60)),
                                      sample=sample, population="all")
        summary_rows.append(s)
        hist_rows.extend(h)

keep = (granule_adata.obs[complexity_col] >= MIN_UNIQUE_GENES).to_numpy()
if MAX_GRANULES is not None:
    idx = np.flatnonzero(keep)
    rng = np.random.default_rng(0)
    keep = np.zeros_like(keep)
    keep[rng.choice(idx, size=min(MAX_GRANULES, idx.size), replace=False)] = True
    print(f"DRY RUN: capped at {keep.sum()} granules")

for sample in C.SAMPLES:
    m = (granule_adata.obs["batch"] == C.dataset(sample)).to_numpy()
    for measure, col in [("n_reads", "n_reads"), ("n_genes", complexity_col),
                         ("sphere_r", "sphere_r"), ("size", "size")]:
        s, h = A2.record_distribution(granule_adata.obs.loc[m & keep, col].to_numpy(), measure,
                                      C.HIST_BINS.get(measure, (0.0, 60.0, 60)),
                                      sample=sample, population="multigene")
        summary_rows.append(s)
        hist_rows.extend(h)

pd.DataFrame(summary_rows).to_csv(OUT / "complexity_summary.csv", index=False)
pd.DataFrame(hist_rows).to_parquet(OUT / "complexity_histogram.parquet", index=False)

In [6]:
# Retention -- the number the response letter has to quote.
obs = granule_adata.obs
ret = (obs.assign(keep=keep)
          .groupby(["batch", "brain_area"], observed=True)["keep"]
          .agg(n_all="size", n_multigene="sum").reset_index())
ret["retention"] = ret["n_multigene"] / ret["n_all"]

overall = (obs.assign(keep=keep).groupby("batch", observed=True)["keep"]
              .agg(n_all="size", n_multigene="sum").reset_index())
overall["brain_area"] = "overall"
overall["retention"] = overall["n_multigene"] / overall["n_all"]
ret = pd.concat([overall, ret], ignore_index=True)
ret["min_unique_genes"] = MIN_UNIQUE_GENES
ret.to_csv(OUT / "retention_by_region.csv", index=False)
print(ret[ret["brain_area"] == "overall"])

# GATE: if either sample retains very few granules the >= 3 cutoff is not usable and the
# constant in a2_config.py should drop to 2 before going on.
worst = overall["retention"].min()
print(f"\nworst-sample retention at >= {MIN_UNIQUE_GENES} unique genes: {worst:.1%}")
if worst < 0.20:
    print("WARNING: retention below 20% -- reconsider C.MIN_UNIQUE_GENES before section 2.")

           batch   n_all  n_multigene brain_area  retention  min_unique_genes
0  MERSCOPE_WT_1  681337       467763    overall   0.686537                 3
1  MERSCOPE_AD_1  398809       283504    overall   0.710877                 3

worst-sample retention at >= 3 unique genes: 68.7%


## 2. Re-embedding the subset

The subset is re-normalised and re-embedded from raw counts, exactly as `code/3_detection.py:113-117`
does for the full set. `granule_id` is positional within a sample, so it is carried through the
subset rather than regenerated -- `(batch, granule_id)` stays the join key against every published
artifact.

In [ ]:
adata = granule_adata[keep].copy()
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.tl.pca(adata, n_comps=10, svd_solver="auto")
sc.tl.tsne(adata, n_pcs=10)

adata.obs["sample"] = adata.obs["batch"]
adata.obs["sample_simple"] = adata.obs["batch"].replace(
    {C.dataset("WT"): "WT", C.dataset("AD"): "AD"})
adata.write_h5ad(OUT / "multigene_granule_adata_tsne.h5ad")
print(adata.shape)
print(adata.obs["sample_simple"].value_counts())

## 3. Granule subtyping -- K-means then manual annotation

Same procedure as the published main result (`code/benchmark/benchmark_subtyping.ipynb` cells 15
and 22): `MiniBatchKMeans(k=15, batch_size=5000, n_init=20, random_state=1)` on the 34
compartment markers, **normalised on the full panel first and subset afterwards** -- reversing
that order changes the clustering.

k is held at 15 so the subset's subtype structure is directly comparable to the published one.

In [ ]:
var_names = [g for g in C.REF_GENES if g in adata.var_names]
print(f"{len(var_names)} of {len(C.REF_GENES)} reference markers present")

adata_ref = adata[:, var_names].copy()
A2.run_manual_subtyping(adata_ref, n_clusters=C.K_SUBTYPE, seed=C.SUBTYPE_SEED,
                        batch_size=C.KMEANS_BATCH_SIZE, n_init=C.KMEANS_N_INIT)
adata.obs["granule_subtype_kmeans"] = adata_ref.obs["granule_subtype_kmeans"].values

groupby = "granule_subtype_kmeans"
ax = sc.pl.heatmap(adata_ref, var_names=var_names, groupby=groupby, cmap="Reds",
                   standard_scale="var", dendrogram=False, swap_axes=True, show=False,
                   figsize=(10, 6))
plt.gcf().savefig(OUT / "heatmap_subtype.jpeg", dpi=500, bbox_inches="tight")
plt.close()
print("saved heatmap_subtype.jpeg")

In [ ]:
# Reading aid for the EDIT ME block below. Column-scales the per-cluster means the same way
# `standard_scale="var"` does, so this table and the heatmap show the same thing -- but reading
# the mapping off the table is far less error-prone than eyeballing the figure.
marker_table = A2.top_marker_table(adata_ref, var_names, cluster_column=groupby,
                                   thr=0.5, max_markers=6)
marker_table.to_csv(OUT / "subtype_top_markers.csv", index=False)
with pd.option_context("display.max_colwidth", 200):
    print(marker_table.to_string(index=False))

### `MANUAL_SUBTYPE_MAPPING` -- **EDIT ME**

Fill this in from `heatmap_subtype.jpeg` plus the top-marker table printed above. Compartment
assignment follows `C.MARKER_GENES`: pre-synaptic (Bsn, Syn1, Syp, Syt1, Vamp2, Snap25, Stx1a,
Slc17a7, Slc32a1, Cplx2, Nrxn1, Gap43), post-synaptic (Camk2a, Dlg3/4, Gphn, Gria1/2, Homer1/2,
Nlgn1/2/3, Shank1/3), dendritic (Actb, Cyfip2, Ddn, Map1a, Map2), axonal (Ank3, Nav1, Nfasc,
Mapt, Tubb3).

**These cluster ids are not the published ones.** The subset was clustered separately, so cluster
7 here has nothing to do with cluster 7 in the published seed-1 result. That published mapping is
reproduced below **only** as a worked example of the format:

```
C.PUBLISHED_SUBTYPE_MAPPING_SEED1 = {
    "pre-syn": ["0", "11", "12", "13"],   "post-syn": ["1", "2", "3"],
    "dendrites": ["4", "8"],              "axons": [],
    "pre & post": ["6"],                  "post & den": ["5", "9", "10"],
    "pre & post & den": ["7", "14"],      "others": [],
}
```

Every cluster 0-14 must appear exactly once. `A2.apply_manual_annotation` **raises** on a
misspelt key, an out-of-range id, a duplicate, or an omission -- a stale mapping would otherwise
fail silently and every density and DE number below would inherit the error. Put genuinely
uninterpretable clusters under `"others"` rather than leaving them out.

Leave the dict empty on the first pass: the cell detects that and stops, having already written
the heatmap and the marker table.

In [ ]:
# ================================ EDIT ME ================================ #
# Cluster ids -> compartment, read off heatmap_subtype.jpeg + subtype_top_markers.csv.
# Valid keys: pre-syn, post-syn, dendrites, axons, and any " & " combination of the four,
# plus "others". Every cluster "0".."14" must appear exactly once.
MANUAL_SUBTYPE_MAPPING = {
    "pre-syn": [],
    "post-syn": [],
    "dendrites": [],
    "axons": [],
    "pre & post": [],
    "pre & den": [],
    "post & den": [],
    "pre & post & den": [],
    "others": [],
}
# ========================================================================= #

MAPPING_FILLED = any(len(v) for v in MANUAL_SUBTYPE_MAPPING.values())
if not MAPPING_FILLED:
    print("MANUAL_SUBTYPE_MAPPING is empty -- first pass.\n"
          f"Read {OUT / 'heatmap_subtype.jpeg'} and subtype_top_markers.csv, fill the dict "
          "above, then rerun from section 3.")

In [ ]:
if MAPPING_FILLED:
    A2.apply_manual_annotation(adata, MANUAL_SUBTYPE_MAPPING, n_clusters=C.K_SUBTYPE,
                               cluster_column=groupby)
    adata_ref.obs["granule_subtype_manual"] = adata.obs["granule_subtype_manual"].values
    adata_ref.obs["granule_subtype_manual_simple"] = adata.obs["granule_subtype_manual_simple"].values
    for a in (adata, adata_ref):
        a.obs["granule_subtype_manual_simple"] = pd.Categorical(
            a.obs["granule_subtype_manual_simple"], categories=C.SUBTYPE_ORDER, ordered=True
        ).remove_unused_categories()

    composition = (adata.obs["granule_subtype_manual_simple"]
                   .value_counts(normalize=True).rename("fraction").reset_index())
    composition.columns = ["subtype", "fraction"]
    composition.to_csv(OUT / "subtype_composition.csv", index=False)
    print(composition.to_string(index=False))

    # Verification heatmap: clusters re-ordered by compartment. Read it as a verdict on the
    # mapping, not as a new result -- blocks that do not hold together mean the mapping is wrong.
    order = A2.ordered_cluster_ids(MANUAL_SUBTYPE_MAPPING, C.K_SUBTYPE, C.SUBTYPE_ORDER)
    adata_ref.obs[groupby] = pd.Categorical(adata_ref.obs[groupby].astype(str),
                                            categories=order, ordered=True)
    ax = sc.pl.heatmap(adata_ref, var_names=var_names, groupby=groupby, cmap="Reds",
                       standard_scale="var", dendrogram=False, swap_axes=True, show=False,
                       figsize=(10, 6.15))
    plt.gcf().savefig(OUT / "heatmap_subtype_ordered.jpeg", dpi=500, bbox_inches="tight")
    plt.close()
    adata_ref.obs[groupby] = pd.Categorical(adata_ref.obs[groupby].astype(str),
                                            categories=[str(i) for i in range(C.K_SUBTYPE)],
                                            ordered=True)

    pd.DataFrame([{"min_unique_genes": MIN_UNIQUE_GENES, "complexity_col": complexity_col,
                   "subtype_seed": C.SUBTYPE_SEED, "k_subtype": C.K_SUBTYPE,
                   "n_granules": int(adata.n_obs),
                   "mapping": json.dumps(MANUAL_SUBTYPE_MAPPING)}]
                 ).to_csv(OUT / "run_info.csv", index=False)

In [ ]:
if MAPPING_FILLED:
    # t-SNE panels, same rendering as the published main result.
    sc.set_figure_params(figsize=(8, 8))
    ax = sc.pl.tsne(adata, color="sample_simple",
                    palette={"WT": C.WT_COLOR, "AD": C.AD_COLOR}, size=1, show=False)
    for a in [ax]:
        a.grid(False); a.set_xticks([]); a.set_yticks([])
        a.set_xlabel(""); a.set_ylabel(""); a.set_title("")
        for spine in a.spines.values():
            spine.set_visible(False)
    plt.gcf().savefig(OUT / "granules_by_batch_tsne.jpeg", dpi=500, bbox_inches="tight")
    plt.close()

    for col in ["granule_subtype_kmeans", "granule_subtype_manual", "granule_subtype_manual_simple"]:
        sc.set_figure_params(figsize=(8, 8))
        ax = sc.pl.embedding(adata, basis="tsne", color=col, size=1, show=False)
        ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(""); ax.set_ylabel(""); ax.set_title("")
        for spine in ax.spines.values():
            spine.set_visible(False)
        plt.gcf().savefig(OUT / f"{col}_tsne.jpeg", dpi=500, bbox_inches="tight")
        plt.close()
    print("saved t-SNE panels")

## 4. Granule density, WT vs AD

Per-(sample, brain area, subtype) density = mean granules per 50 um spot, with a 500-replicate
bootstrap CI and a t-test on `log1p` per-spot counts, Bonferroni-corrected within subtype and
BH-corrected across all tests -- the procedure from `benchmark_subtyping.ipynb` cell 22.

Each sample's granules are overlaid on **its own** `spots.h5ad` using the raw per-sample
coordinates (`obs["global_x"]` here is `sphere_x`), never the aligned combined frame. AD counts
are divided by `CAPTURE_EFFICIENCY_COEF`, as in the published table.

The output schema matches `subtype_density_per_region_granule_adata_tsne.csv` exactly, so
`A2_figures.R` reuses the published bar-plot code unchanged.

In [ ]:
if MAPPING_FILLED:
    per_spot = {}
    for sample in C.SAMPLES:
        spots_s = sc.read_h5ad(C.spots_path(sample))
        obs_s = adata.obs[adata.obs["sample_simple"] == sample]
        per_spot[sample] = A2.per_spot_counts_table(
            obs_s, spots_s, subtype_col="granule_subtype_manual_simple", sample_label=sample,
            area_col="brain_area", coord_keys=("global_x", "global_y"), grid_len=C.SPOT_GRID)
        print(sample, per_spot[sample].shape)

    per_spot["AD"]["count"] = per_spot["AD"]["count"] / C.CAPTURE_EFFICIENCY_COEF

    density_df = A2.density_from_per_spot(pd.concat(per_spot.values(), ignore_index=True))
    density_df["setting"] = f"multigene_min{MIN_UNIQUE_GENES}"
    density_df = A2.add_density_significance(density_df, per_spot["WT"], per_spot["AD"],
                                             n_bootstrap=C.N_BOOTSTRAP)
    cols = ["sample", "brain_area", "subtype", "density", "n_spots", "setting", "density_sd",
            "density_sem", "density_ci_low", "density_ci_high", "p_val", "p_bonf", "q_val",
            "p_val_star", "p_bonf_star", "q_val_star"]
    density_df = density_df[cols]
    density_df.to_csv(OUT / "subtype_density_per_region_multigene.csv", index=False)

    keep_cols = ["granule_id", "sample", "granule_subtype_manual_simple", "global_x", "global_y",
                 "granule_subtype_kmeans", "granule_subtype_manual", "n_reads", complexity_col]
    adata.obs[[c for c in keep_cols if c in adata.obs.columns]].to_parquet(
        OUT / "granule_subtype_labels_multigene.parquet", index=False)

    print(density_df[density_df["brain_area"] == C.ROI].to_string(index=False))

## 5. Neuropil microdomains

The published microdomain analysis (`code/7_neuropil_subdomains.ipynb` cell 9), rerun with the
multi-gene subset swapped in for the granules and **nothing else changed**:

* the 50 um Isocortex spot grid and its SpaGCN `layer_labels` are **read** from
  `neuropil_subdomains_spots_ambient.h5ad`, not recomputed -- recomputing the scaffold along with
  the granules would confound the two changes;
* `subdomain_kmeans` **is** recomputed, because microdomains are *defined* by granule-subtype
  composition; inheriting the published labels would be circular;
* every setting stays at the published `neuropil_subdomains_Isocortex_50` values (ROI Isocortex,
  50 um spots, K_subdomain = 4, hard embedding, gaussian smoothing).

Coordinates come from `neuropil_subdomains_granule_adata.h5ad`, the only artifact whose
`global_x/global_y` are aligned to the spot grid.

In [ ]:
if MAPPING_FILLED:
    OUT_SUB = OUT / f"neuropil_subdomains_{C.ROI}_{C.SPOT_GRID}"
    OUT_SUB.mkdir(parents=True, exist_ok=True)

    # Grid-aligned coordinates + counts layer, restricted to the retained granules and carrying
    # this analysis's subtype labels rather than the published ones.
    granule_sub = sc.read_h5ad(C.SUBDOMAIN_GRANULE_ADATA)
    key_all = pd.MultiIndex.from_arrays(
        [granule_sub.obs["batch"].astype(str), granule_sub.obs["granule_id"].astype(str)])
    key_keep = pd.MultiIndex.from_arrays(
        [adata.obs["batch"].astype(str), adata.obs["granule_id"].astype(str)])
    assert key_keep.is_unique, "(batch, granule_id) is not unique in the subset"
    granule_sub = granule_sub[key_all.isin(key_keep)].copy()

    lab = (adata.obs[["batch", "granule_id", "granule_subtype_kmeans",
                      "granule_subtype_manual_simple"]]
           .astype({"batch": str, "granule_id": str}))
    obs = (granule_sub.obs.astype({"batch": str, "granule_id": str})
           .merge(lab, on=["batch", "granule_id"], how="left", validate="one_to_one",
                  suffixes=("_published", "")))
    obs.index = granule_sub.obs.index
    granule_sub.obs = obs
    assert granule_sub.obs["granule_subtype_kmeans"].notna().all(), "subtype join left gaps"
    granule_sub.obs["granule_subtype_kmeans"] = pd.Categorical(
        granule_sub.obs["granule_subtype_kmeans"].astype(str),
        categories=[str(i) for i in range(C.K_SUBTYPE)], ordered=True)
    print(f"{granule_sub.n_obs} multi-gene granules on the spot grid")

    # Scaffold -- read, never recomputed.
    spots = sc.read_h5ad(C.SUBDOMAIN_SPOTS_50)
    adata_cells = sc.read_h5ad(C.SUBDOMAIN_ADATA)
    count_matrix = pd.concat([pd.read_parquet(C.count_matrix_path(s)) for s in C.SAMPLES],
                             axis=0, ignore_index=True)
    count_matrix["cell_id"] = count_matrix["cell_id"].astype(str)
    count_matrix = (count_matrix.set_index("cell_id")
                    .reindex(columns=list(granule_sub.var_names), fill_value=0).reset_index())
    adata_neuron = adata_cells[
        adata_cells.obs["brain_area"].str.startswith(C.ROI)
        & adata_cells.obs["cell_type"].isin(["GABAergic", "Glutamatergic"])].copy()
    print(spots.shape, adata_neuron.shape, count_matrix.shape)

In [ ]:
if MAPPING_FILLED:
    spots0 = spots.copy()
    embeddings, embeddings_features, aux_features, spot_granule_expression, spot_cell_expression = \
        spot_embedding(
            spots=spots0,
            granule_adata=granule_sub,
            adata=adata_neuron,
            count_matrix=count_matrix,
            spot_loc_key=("global_x", "global_y"),
            spot_width=C.SPOT_GRID,
            spot_height=C.SPOT_GRID,
            granule_loc_key=("global_x", "global_y"),
            granule_subtype_key="granule_subtype_kmeans",
            subtype_names=[str(i) for i in range(C.K_SUBTYPE)],
            granule_count_layer="counts",
            cell_loc_key=("global_x", "global_y"),
            cell_id_key="cell_id",
            count_matrix_cell_id_key="cell_id",
            include_soma_features=True,
            smoothing=True,
            smoothing_radius=np.sqrt(2) * C.SPOT_GRID + 1,
            smoothing_mode="gaussian",
        )
    for aux_key, aux_val in aux_features.items():
        spots0.obs[aux_key] = aux_val

    spot_ambient_expression = np.maximum(
        spots0.layers["extrasomatic_transcripts"] - spot_granule_expression, 0)

    mask = (spots0.obs["granule_count"] > 0).to_numpy()
    spots0 = spots0[mask].copy()
    embeddings = embeddings[mask].copy()
    spot_granule_expression = spot_granule_expression[mask].copy()
    spot_cell_expression = spot_cell_expression[mask].copy()
    spot_ambient_expression = spot_ambient_expression[mask].copy()
    print(f"{spots0.n_obs} spots with >= 1 multi-gene granule "
          f"(published run kept 4310 spots of {len(mask)})")

    row_sums = embeddings.sum(axis=1, keepdims=True)
    X_spot = np.divide(embeddings, row_sums, out=np.zeros_like(embeddings, dtype=float),
                       where=row_sums > 0)

In [ ]:
if MAPPING_FILLED:
    # K sweep, for the record -- the published run chose K = 4 from this.
    results = []
    for k in C.SUBDOMAIN_K_RANGE:
        km = MiniBatchKMeans(n_clusters=k, random_state=C.SUBDOMAIN_SEED,
                             batch_size=C.SUBDOMAIN_BATCH_SIZE)
        labels = km.fit_predict(X_spot)
        label_list = [labels]
        for seed in C.STABILITY_SEEDS[1:]:
            label_list.append(MiniBatchKMeans(n_clusters=k, random_state=seed,
                                              batch_size=C.SUBDOMAIN_BATCH_SIZE
                                              ).fit_predict(X_spot))
        ari = [adjusted_rand_score(label_list[i], label_list[j])
               for i in range(len(label_list)) for j in range(i + 1, len(label_list))]
        results.append({"n_clusters": k, "inertia": km.inertia_,
                        "ari_stability_mean": float(np.mean(ari)) if ari else np.nan})
    pd.DataFrame(results).to_csv(
        OUT_SUB / "hard_normalized_benchmark_clustering_results.csv", index=False)
    print(pd.DataFrame(results).to_string(index=False))

In [ ]:
if MAPPING_FILLED:
    n_clusters = C.K_SUBDOMAIN

    lda = LatentDirichletAllocation(n_components=n_clusters, random_state=C.SUBDOMAIN_SEED)
    spots0.obs["subdomain_lda"] = [f"Subdomain {l + 1}"
                                   for l in np.argmax(lda.fit_transform(X_spot), axis=1)]
    gmm = GaussianMixture(n_components=n_clusters, random_state=C.SUBDOMAIN_SEED)
    spots0.obs["subdomain_gmm"] = [f"Subdomain {l + 1}" for l in gmm.fit_predict(X_spot)]
    kmeans = KMeans(n_clusters=n_clusters, random_state=C.SUBDOMAIN_SEED, n_init=20)
    spots0.obs["subdomain_kmeans"] = [f"Subdomain {l + 1}" for l in kmeans.fit_predict(X_spot)]
    mbk = MiniBatchKMeans(n_clusters=n_clusters, batch_size=C.SUBDOMAIN_BATCH_SIZE,
                          random_state=C.SUBDOMAIN_SEED, n_init=20)
    spots0.obs["subdomain_minibatch"] = [f"Subdomain {l + 1}" for l in mbk.fit_predict(X_spot)]

    # NOTE: the published run applied a cosmetic relabel_map so that "Subdomain 1" was the
    # pre-synaptic-rich one. That map is specific to the published clustering and is NOT applied
    # here -- these labels are raw K-means output, which is exactly why the pair to contrast is a
    # manual input below.
    spots0.obs["subdomain_kmeans"] = pd.Categorical(
        spots0.obs["subdomain_kmeans"],
        categories=[f"Subdomain {i + 1}" for i in range(n_clusters)], ordered=True)

    spots0.obs[["spot_id", "layer_labels", "subdomain_lda", "subdomain_gmm", "subdomain_kmeans",
                "subdomain_minibatch"]].to_parquet(
        OUT_SUB / f"{n_clusters}_hard_normalized_cluster_labels.parquet")
    print(spots0.obs["subdomain_kmeans"].value_counts().to_string())

In [ ]:
if MAPPING_FILLED:
    # Maps and composition heatmaps, same rendering as the published run.
    ordered_clusters = A2.ordered_cluster_ids(MANUAL_SUBTYPE_MAPPING, C.K_SUBTYPE, C.SUBTYPE_ORDER)
    cluster_to_cat = {}
    for subtype, clusters in MANUAL_SUBTYPE_MAPPING.items():
        simple = "mixed" if " & " in str(subtype) else str(subtype)
        for c in clusters:
            cluster_to_cat[str(c)] = simple
    ordered_cats = ["pre-syn", "post-syn", "dendrites", "axons", "mixed", "others"]
    cat_to_color = dict(zip(ordered_cats, sns.color_palette("Set2", len(ordered_cats))))

    for cluster_col in ["subdomain_lda", "subdomain_gmm", "subdomain_kmeans", "subdomain_minibatch"]:
        spots0.uns.pop(f"{cluster_col}_colors", None)
        short = cluster_col.split("_")[1]
        sc.set_figure_params(scanpy=True, figsize=C.PLOT_FIGSIZE)
        ax = sc.pl.scatter(spots0, x="global_x", y="global_y", color=cluster_col,
                           size=C.PLOT_SPOT_SIZE, title=" ", show=False)
        ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(""); ax.set_ylabel(""); ax.set_title("")
        for spine in ax.spines.values():
            spine.set_visible(False)
        plt.savefig(OUT_SUB / f"{n_clusters}_hard_normalized_{short}.jpeg", dpi=500,
                    bbox_inches="tight")
        plt.close()

        for cluster in spots0.obs[cluster_col].unique():
            sc.set_figure_params(scanpy=True, figsize=C.PLOT_FIGSIZE)
            ax = sc.pl.scatter(spots0[spots0.obs[cluster_col] == cluster], x="global_x",
                               y="global_y", color=cluster_col, size=C.PLOT_SPOT_SIZE,
                               title=" ", show=False)
            ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
            ax.set_xlabel(""); ax.set_ylabel(""); ax.set_title("")
            for spine in ax.spines.values():
                spine.set_visible(False)
            plt.savefig(OUT_SUB / f"{n_clusters}_hard_normalized_{short}_{cluster}.jpeg",
                        dpi=500, bbox_inches="tight")
            plt.close()

        desired_order = [f"Subdomain {i + 1}" for i in range(n_clusters)]
        embedding_df = pd.DataFrame(X_spot, columns=[str(i) for i in range(C.K_SUBTYPE)])
        embedding_df[cluster_col] = spots0.obs[cluster_col].values
        cluster_means = embedding_df.groupby(cluster_col).mean().reindex(desired_order)
        col_order = [s for s in ordered_clusters if s in cluster_means.columns]
        cluster_means = cluster_means[col_order]
        col_colors = pd.Series([cluster_to_cat.get(s, "others") for s in col_order],
                               index=col_order, name="Category").map(cat_to_color)

        for mat, cmap, kw, stem in [
                (cluster_means, "Reds", {}, "heatmap"),
                (np.log2((cluster_means + 1e-8).divide(
                    embedding_df[col_order].mean(axis=0) + 1e-8, axis=1)), "bwr",
                 {"center": 0}, "heatmap_log2fc")]:
            g = sns.clustermap(mat, cmap=cmap, annot=False, col_colors=col_colors,
                               row_cluster=False, col_cluster=False, linewidths=0.5,
                               linecolor="lightgray", xticklabels=True, yticklabels=True,
                               figsize=(10, 2.5 if stem == "heatmap" else 3),
                               cbar_pos=(0.02, 0.2, 0.02, 0.5), **kw)
            g.ax_heatmap.set_position([0.1, 0.2, 0.75, 0.7])
            g.cax.set_position([0.88, 0.2, 0.02, 0.5])
            g.ax_col_colors.set_position([0.1, 0.92, 0.75, 0.04])
            g.ax_heatmap.tick_params(axis="both", which="both", length=0)
            g.ax_heatmap.grid(False)
            g.ax_heatmap.set_xlabel("Granule Subtype")
            g.ax_heatmap.set_ylabel("Neuropil Subdomain", labelpad=10)
            g.ax_heatmap.yaxis.set_label_position("left")
            g.ax_heatmap.yaxis.tick_left()
            g.ax_heatmap.set_title(" ")
            g.ax_row_dendrogram.set_visible(False)
            g.ax_col_dendrogram.set_visible(False)
            for spine in g.ax_col_colors.spines.values():
                spine.set_visible(False)
            g.cax.grid(False)
            g.ax_col_colors.set_xticks([]); g.ax_col_colors.set_yticks([])
            g.ax_col_colors.grid(False)
            g.ax_heatmap.spines["top"].set_visible(False)
            plt.savefig(OUT_SUB / f"{n_clusters}_hard_normalized_{stem}_{short}.jpeg",
                        dpi=500, bbox_inches="tight")
            plt.close()
    print("saved subdomain maps and heatmaps to", OUT_SUB)

### `SUBDOMAIN_PAIRS` -- **EDIT ME**

The published DE contrast was `Subdomain 1 vs Subdomain 2`, but that pairing depended on a
cosmetic `relabel_map` fitted to the published clustering (`7_neuropil_subdomains.ipynb` cell 9),
which does **not** transfer. K-means labels here are arbitrary, so the pre-synaptic-rich domain
and the domain to contrast it against **are not necessarily 1 and 2**.

Pick the pair after looking at:

* `4_hard_normalized_kmeans.jpeg` and the per-subdomain maps -- which domain occupies the
  superficial/neuropil-rich cortical position;
* `4_hard_normalized_heatmap_log2fc_kmeans.jpeg` -- which domain is enriched for the pre-synaptic
  clusters and which for the post-synaptic ones.

Several pairs are allowed; each produces its own granule / cell / ambient DE triple, named exactly
as in the published `neuropil_subdomains_Isocortex_50/`, so `A2_figures.R` discovers and
GSEA-scores them without repointing.

The placeholder below is the two largest subdomains by spot count, so the first pass runs end to
end -- **replace it before quoting anything.**

In [ ]:
if MAPPING_FILLED:
    _sizes = spots0.obs["subdomain_kmeans"].value_counts()
    print("subdomain sizes:\n", _sizes.to_string())

    # ================================ EDIT ME ================================ #
    # (target, reference) pairs to contrast. Placeholder = the two largest subdomains.
    SUBDOMAIN_PAIRS = [(str(_sizes.index[0]), str(_sizes.index[1]))]
    # e.g. SUBDOMAIN_PAIRS = [("Subdomain 3", "Subdomain 1")]
    # ========================================================================= #

    print("contrasting:", SUBDOMAIN_PAIRS)

In [ ]:
if MAPPING_FILLED:
    cluster_col = "subdomain_kmeans"
    for target_cluster, reference_cluster in SUBDOMAIN_PAIRS:
        m = spots0.obs[cluster_col].isin([target_cluster, reference_cluster]).to_numpy()
        obs_select = spots0.obs.loc[m].copy()
        for expr, label in zip([spot_granule_expression[m], spot_cell_expression[m],
                                spot_ambient_expression[m]], ["granule", "cell", "ambient"]):
            ad_de = anndata.AnnData(X=expr.copy(), obs=obs_select.copy(),
                                    var=granule_sub.var.copy())
            sc.pp.normalize_total(ad_de, target_sum=1e4)
            sc.pp.log1p(ad_de)
            sc.tl.rank_genes_groups(ad_de, groupby=cluster_col, groups=[target_cluster],
                                    reference=reference_cluster, method="wilcoxon")
            df = sc.get.rank_genes_groups_df(ad_de, group=target_cluster)
            df = df.sort_values("logfoldchanges", ascending=False)
            df.to_csv(OUT_SUB / f"{label}_DE_genes_{target_cluster}_vs_{reference_cluster}.csv",
                      index=False)
        print(f"wrote DE for {target_cluster} vs {reference_cluster}")

## 6. Read-count stratification

The other half of the reviewer's request: show the findings are not confined to the lowest-count
granules. This runs over **all** granules, independent of the multi-gene subset, splitting them
into read-count terciles and recomputing subtype composition and WT/AD subtype density within
each stratum with the same test as section 4.

Deliberately lighter than section 5 -- no microdomain rerun per tercile. Subtype labels are the
**published** ones (`granule_subtype_labels_granule_adata_tsne.parquet`), so this section measures
the effect of read depth alone, with the clustering held fixed.

In [ ]:
published_labels = pd.read_parquet(C.COMBINED_SUBTYPE_LABELS)
strata_obs = granule_adata.obs[["batch", "granule_id", "brain_area", "global_x", "global_y",
                                "n_reads", "n_genes_panel", "n_genes_nonNC"]].copy()
strata_obs = strata_obs.astype({"batch": str, "granule_id": str}).merge(
    published_labels[["sample", "granule_id", "granule_subtype_manual_simple"]]
    .astype({"sample": str, "granule_id": str}),
    left_on=["batch", "granule_id"], right_on=["sample", "granule_id"],
    how="left", validate="one_to_one")
assert strata_obs["granule_subtype_manual_simple"].notna().all(), "published-label join left gaps"

strata_obs["read_tercile"] = A2.read_terciles(strata_obs["n_reads"].to_numpy(),
                                              labels=C.READ_TERCILE_LABELS)
strata_obs["sample_simple"] = strata_obs["batch"].replace(
    {C.dataset("WT"): "WT", C.dataset("AD"): "AD"})

edges = (strata_obs.groupby("read_tercile")["n_reads"]
         .agg(n="size", min="min", max="max", median="median").reset_index())
edges.to_csv(OUT_STRATA / "readstrata_edges.csv", index=False)
print(edges.to_string(index=False))

In [ ]:
# Subtype composition and complexity within each tercile.
comp_rows = (strata_obs.groupby(["sample_simple", "read_tercile",
                                 "granule_subtype_manual_simple"], observed=True)
             .size().rename("n").reset_index())
comp_rows["fraction"] = comp_rows["n"] / comp_rows.groupby(
    ["sample_simple", "read_tercile"], observed=True)["n"].transform("sum")

cplx_rows = (strata_obs.groupby(["sample_simple", "read_tercile"], observed=True)
             .agg(n_granules=("n_reads", "size"),
                  median_reads=("n_reads", "median"),
                  median_unique_genes=("n_genes_nonNC", "median"),
                  frac_multigene=("n_genes_nonNC",
                                  lambda s: float((s >= MIN_UNIQUE_GENES).mean()))
                  ).reset_index())
comp_rows.to_csv(OUT_STRATA / "readstrata_summary.csv", index=False)
cplx_rows.to_csv(OUT_STRATA / "readstrata_complexity.csv", index=False)
print(cplx_rows.to_string(index=False))

In [ ]:
# WT vs AD subtype density within each read tercile.
density_parts = []
spots_by_sample = {s: sc.read_h5ad(C.spots_path(s)) for s in C.SAMPLES}
for tercile in C.READ_TERCILE_LABELS:
    per_spot_t = {}
    for sample in C.SAMPLES:
        sel = strata_obs[(strata_obs["sample_simple"] == sample)
                         & (strata_obs["read_tercile"] == tercile)]
        per_spot_t[sample] = A2.per_spot_counts_table(
            sel, spots_by_sample[sample], subtype_col="granule_subtype_manual_simple",
            sample_label=sample, area_col="brain_area",
            coord_keys=("global_x", "global_y"), grid_len=C.SPOT_GRID)
    per_spot_t["AD"]["count"] = per_spot_t["AD"]["count"] / C.CAPTURE_EFFICIENCY_COEF

    d = A2.density_from_per_spot(pd.concat(per_spot_t.values(), ignore_index=True))
    d["setting"] = f"read_tercile_{tercile}"
    d = A2.add_density_significance(d, per_spot_t["WT"], per_spot_t["AD"],
                                    n_bootstrap=C.N_BOOTSTRAP)
    d["read_tercile"] = tercile
    density_parts.append(d)
    print(f"tercile {tercile}: done")

readstrata_density = pd.concat(density_parts, ignore_index=True)
readstrata_density.to_csv(OUT_STRATA / "readstrata_density.csv", index=False)
print(readstrata_density[(readstrata_density["brain_area"] == C.ROI)
                         & (readstrata_density["subtype"] == "pre-syn")].to_string(index=False))

## 7. Correctness gates

Off by default (`VALIDATE = False` in section 0). These check the claims the analysis rests on,
not the biology.

In [ ]:
if VALIDATE:
    # (a) Subsetting rows == re-profiling the retained spheres. profile() queries each sphere
    #     independently, so this must hold exactly -- it is what licenses skipping detection.
    from mcDETECT.model import mcDETECT
    rng = np.random.default_rng(0)
    sample = "WT"
    tx = pd.read_parquet(C.transcripts_path(sample))
    genes_csv = list(pd.read_csv(C.genes_path(sample)).iloc[:, 0])
    gran = pd.read_parquet(C.mcdetect_granules_path(sample))
    pick = rng.choice(gran.shape[0], size=1000, replace=False)
    mc = mcDETECT(transcripts=tx, gnl_genes=C.SYN_GENES, nc_genes=None,
                  **C.DETECT_KWARGS_ROUGH)
    reprofiled = mc.profile(gran.iloc[pick].reset_index(drop=True), genes=genes_csv)

    full = sc.read_h5ad(C.mcdetect_granule_adata_path(sample))
    ref_counts = full.layers["counts"][pick][:, [genes_csv.index(g) for g in reprofiled.var_names]]
    diff = abs(reprofiled.X - ref_counts).max()
    print(f"(a) subset == re-profile: max abs difference = {diff}")
    assert diff == 0, "subsetting is not equivalent to re-profiling -- do not skip detection"

    # (b) The vectorised density primitive matches the published per-spot loop.
    spots_wt = sc.read_h5ad(C.spots_path("WT"))
    sxy = spots_wt.obs[["global_x", "global_y"]].to_numpy(float)
    gxy = granule_adata.obs.loc[granule_adata.obs["batch"] == C.dataset("WT"),
                                ["global_x", "global_y"]].to_numpy(float)[:20000]
    fast = A2.spot_counts(gxy, sxy, C.SPOT_GRID)
    slow = A2._spot_counts_reference(gxy, sxy, C.SPOT_GRID)
    print(f"(b) spot_counts == published loop: {np.array_equal(fast, slow)}")
    assert np.array_equal(fast, slow)

    # (c) `comp` is a marker count, not a gene count -- the finding behind this whole design.
    print(f"(c) comp max = {int(granule_adata.obs['comp'].max())} "
          f"(<= {len(C.SYN_GENES)} markers); disagrees with {complexity_col} for "
          f"{(granule_adata.obs['comp'].astype(int) != granule_adata.obs[complexity_col].astype(int)).mean():.1%} "
          f"of granules")
    assert granule_adata.obs["comp"].max() <= len(C.SYN_GENES)

    # (d) The spot scaffold was inherited, not recomputed.
    if MAPPING_FILLED:
        ref_spots = sc.read_h5ad(C.SUBDOMAIN_SPOTS_50)
        assert set(spots0.obs["spot_id"]).issubset(set(ref_spots.obs["spot_id"]))
        merged = spots0.obs[["spot_id", "layer_labels"]].merge(
            ref_spots.obs[["spot_id", "layer_labels"]], on="spot_id", suffixes=("", "_ref"))
        assert (merged["layer_labels"].astype(str)
                == merged["layer_labels_ref"].astype(str)).all()
        print("(d) spot grid and layer_labels identical to the published scaffold")